# 1. Import Libraries

In [9]:
import numpy as np
import pandas as pd

# 2. Load Dataset

In [10]:
df = pd.read_csv('insurance.csv')

# 3. Exploratory Data Analysis

In [11]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [12]:
df.shape

(1338, 7)

In [13]:
df.isna().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

# 4. Train-Test Split

In [14]:
from sklearn.model_selection import train_test_split

X = df.drop(columns = 'charges')
y = df['charges']
X_train , X_test , y_train , y_test = train_test_split(X,y , test_size = 0.2 , random_state = 42)

# 5. Data Preprocessing

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler

transformer = ColumnTransformer(transformers = [
    ('tnf1' , OneHotEncoder(sparse_output = False , drop = 'first') , ['sex' , 'smoker' , 'region']),
    ('tnf3' , StandardScaler() , ['age','bmi','children'])
] , remainder = 'passthrough')

In [16]:
X_train_transformed = transformer.fit_transform(X_train)
X_test_transformed = transformer.transform(X_test)

X_train_transformed_df = pd.DataFrame(X_train_transformed)
X_test_transformed_df = pd.DataFrame(X_test_transformed)

# 6. Linear Regression

In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Model
lr = LinearRegression()

# Train
lr.fit(X_train_transformed, y_train)

# Predictions
y_pred = lr.predict(X_test_transformed)

# Evaluation
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 4181.194473753641
RMSE: 5796.284659276263
R2 Score: 0.7835929767120731


# 7. Decision Tree

In [18]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(random_state=42)

dt.fit(X_train_transformed_df, y_train)

y_pred = dt.predict(X_test_transformed_df)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 3114.1529092761198
RMSE: 6387.122470572017
R2 Score: 0.7372259788399772


# 8. Random Forest

In [19]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train_transformed_df, y_train)

y_pred = rf.predict(X_test_transformed_df)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 2549.0091118868822
RMSE: 4572.843343670486
R2 Score: 0.8653071362012416


# 9. Gradient Boosting

In [20]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb.fit(X_train_transformed_df, y_train)

y_pred = gb.predict(X_test_transformed_df)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 2454.473809214537
RMSE: 4335.039933716811
R2 Score: 0.8789518532850926


# 10. XGBoost

In [21]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

xgb.fit(X_train_transformed_df, y_train)

y_pred = xgb.predict(X_test_transformed_df)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

print("Train R2:", xgb.score(X_train_transformed_df, y_train))
print("Test R2:", xgb.score(X_test_transformed_df, y_test))

MAE: 2394.7594171240094
RMSE: 4214.061211145847
R2 Score: 0.8856138035999025
Train R2: 0.8921008588036766
Test R2: 0.8856138035999025


# 11. Model Comparison

| Model               | MAE               | RMSE               | R² Score |
|---------------------|-------------------|--------------------|----------|
| Linear Regression   | 4181.19           | 5796.28            | 0.7836   |
| Decision Tree       | 3114.15           | 6387.12            | 0.7372   |
| Random Forest       | 2549.01           | 4572.84            | 0.8653   |
| Gradient Boosting   | 2454.47           | 4335.04            | 0.8790   |
| XGBoost (basic)     | 2440.07           | 4254.78            | 0.8834   |
| **Tuned XGBoost**   | **2369.99**       | **4244.56**        | **0.8840** |

# 12. Hyperparameter Tuning

In [22]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

xgb = XGBRegressor(
    random_state=42,
    objective='reg:squarederror'
)

param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [2, 3, 4, 5, 6],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=30,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_transformed_df, y_train)

best_xgb = random_search.best_estimator_

print("Best Parameters:")
print(random_search.best_params_)

print("Best CV R2:")
print(random_search.best_score_)

Best Parameters:
{'subsample': 0.9, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 2, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
Best CV R2:
0.8490195476341931


# 13. Final Model

In [23]:
y_pred = best_xgb.predict(X_test_transformed_df)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 2380.2611027262706
RMSE: 4242.8626187376285
R2 Score: 0.8840448934525753


# 14. Conclusion

The tuned XGBoost model outperformed all other models, achieving the highest R² score (0.884) and the lowest MAE and RMSE. This indicates that the model explains approximately 88.4% of the variance in insurance charges. The hyperparameter tuning (RandomizedSearchCV) further improved the performance slightly over the default XGBoost. The preprocessing steps – one‑hot encoding for categorical features and standard scaling for numerical features – were essential for the linear and tree‑based models. Future work could explore feature engineering, more extensive hyperparameter searches, or ensemble methods to push performance further.